# Patrón Estructural: Adapter — Cafetería

**Dominio propio:** integrar una báscula de café de un proveedor externo a la interfaz que espera el sistema de la cafetería.

## Introducción — qué problema resuelve
El sistema de la cafetería pesa el café con una interfaz interna: `Bascula.pesar_gramos()` devuelve los gramos como entero. Compramos una báscula nueva de un proveedor cuyo SDK **no** tiene ese método: expone `read_weight()` y devuelve el peso en **onzas como float**. No puedo (ni quiero) modificar el SDK del proveedor.

El patrón **Adapter** traduce la interfaz del proveedor a la que el sistema espera, sin tocar el código de terceros ni el código cliente.

## Sin patrón (el problema es evidente)

El código cliente (`preparar_dosis`) espera un objeto con `pesar_gramos()`, pero el SDK del proveedor solo tiene `read_weight()` (en onzas). Las interfaces son incompatibles.

In [1]:
class BasculaProveedor:
    """SDK de terceros: NO se puede modificar. Devuelve onzas (float)."""
    def __init__(self, onzas: float) -> None:
        self._onzas = onzas

    def read_weight(self) -> float:
        return self._onzas


def preparar_dosis(bascula, gramos_objetivo: int) -> None:
    # El cliente espera .pesar_gramos() en gramos enteros
    gramos = bascula.pesar_gramos()
    estado = "OK" if gramos >= gramos_objetivo else "FALTA CAFE"
    print(f"Peso: {gramos}g / objetivo {gramos_objetivo}g -> {estado}")


bascula = BasculaProveedor(onzas=0.75)
try:
    preparar_dosis(bascula, 18)  # falla: BasculaProveedor no tiene pesar_gramos()
except AttributeError as e:
    print("Error esperado:", e)

Error esperado: 'BasculaProveedor' object has no attribute 'pesar_gramos'


### Problema visible
`preparar_dosis` llama a `pesar_gramos()`, pero el SDK solo ofrece `read_weight()` (además en onzas). Sin adaptación, la integración lanza `AttributeError`.

## Con patrón Adapter (problema resuelto)

Defino la interfaz que el sistema espera (`Bascula`) y un `BasculaAdapter` que envuelve el SDK del proveedor, expone `pesar_gramos()` y por dentro traduce onzas → gramos.

In [2]:
from abc import ABC, abstractmethod

class Bascula(ABC):
    """Interfaz objetivo que espera el sistema de la cafeteria."""
    @abstractmethod
    def pesar_gramos(self) -> int: ...


class BasculaAdapter(Bascula):
    """Adaptador: encaja el SDK del proveedor en la interfaz Bascula."""
    ONZA_EN_GRAMOS = 28.3495

    def __init__(self, proveedor: BasculaProveedor) -> None:
        self._proveedor = proveedor

    def pesar_gramos(self) -> int:
        onzas = self._proveedor.read_weight()      # interfaz del proveedor
        return round(onzas * self.ONZA_EN_GRAMOS)   # traduccion al formato esperado


# El mismo cliente, ahora funciona sin cambios
bascula = BasculaProveedor(onzas=0.75)
adaptada = BasculaAdapter(bascula)
preparar_dosis(adaptada, 18)  # 0.75 oz ~= 21 g -> OK

Peso: 21g / objetivo 18g -> OK


### Resultado
`preparar_dosis` no cambió y `BasculaProveedor` no se tocó. El `BasculaAdapter` hace de puente: expone `pesar_gramos()` y traduce onzas a gramos por dentro.

## Diagrama UML (clases reales de este ejemplo)
```plantuml
@startuml
abstract class Bascula {
    + pesar_gramos(): int
}
class BasculaProveedor {
    - _onzas: float
    + read_weight(): float
}
class BasculaAdapter {
    - _proveedor: BasculaProveedor
    + pesar_gramos(): int
}
Bascula <|-- BasculaAdapter
BasculaProveedor <.. BasculaAdapter : adapta
@enduml
```

## ¿Por qué Adapter y no otro patrón estructural?

El problema es de **incompatibilidad de interfaces** entre código que ya existe: el SDK del proveedor tiene una firma (`read_weight` en onzas) y mi sistema espera otra (`pesar_gramos` en gramos). No estoy agregando responsabilidades dinámicas a un objeto (eso sería Decorator), ni simplificando un subsistema complejo detrás de una fachada (Facade), ni controlando el acceso a un objeto (Proxy), ni separando abstracción de implementación desde el diseño (Bridge).

**Adapter** es el patrón exacto cuando quieres reutilizar una clase existente cuya interfaz no coincide con la que necesitas, sin modificar ninguna de las dos partes.